# Preprocesamiento de citas médicas

Ejecuta las celdas en orden desde la raíz del proyecto. Partimos del CSV original, conservamos el orden de sus registros y generamos `data/citas_medicas_preprocesadas.csv` para los notebooks 03, 04 y 05.

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
RUTA_DATOS = Path("data") / "KaggleV2-May-2016.csv"

assert RUTA_DATOS.exists(), f"No se encontró el archivo: {RUTA_DATOS}"

df = pd.read_csv(RUTA_DATOS)

print("Dataset cargado.")
print("Dimensiones", df.shape)

Dataset cargado.
Dimensiones (110527, 14)


# Limpieza de datos

El perfilamiento identificó una edad negativa y cinco citas con fecha anterior al día de su registro. Comparamos días de calendario porque una cita puede agendarse y realizarse el mismo día.

In [3]:
edades_negativas = df[df['Age'] < 0]
print(f"\nCantidad de registros con edad negativa: {len(edades_negativas)}")
df = df[df['Age'] >= 0]

# Convertimos a datetime temporalmente para poder usar .dt.date si las columnas aún son strings
fecha_agendado = pd.to_datetime(df['ScheduledDay']).dt.date
fecha_cita = pd.to_datetime(df['AppointmentDay']).dt.date

inconsistencias_fechas = df[fecha_cita < fecha_agendado]
print(f"Cantidad de registros donde la cita es anterior al registro: {len(inconsistencias_fechas)}")

df = df[fecha_cita >= fecha_agendado]

print("\n--- Dataset limpio ---")
print("Dimensiones finales:", df.shape)


Cantidad de registros con edad negativa: 1
Cantidad de registros donde la cita es anterior al registro: 5

--- Dataset limpio ---
Dimensiones finales: (110521, 14)


# Corregir tipos de datos

Convertimos las fechas a UTC y los indicadores binarios a booleanos. La clase positiva de `No-show` corresponde a una inasistencia.

In [4]:
df['PatientId'] = df['PatientId'].astype('int64')
df['ScheduledDay'] = pd.to_datetime(df['ScheduledDay'], utc=True)
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay'], utc=True)

indicadores = ['Scholarship', 'Hipertension', 'Diabetes', 'Alcoholism', 'SMS_received']
assert df[indicadores].isin([0, 1]).all().all()
assert df['No-show'].isin(['Yes', 'No']).all()
df[indicadores] = df[indicadores].astype(bool)
df['No-show'] = df['No-show'].eq('Yes')
df['Handcap'] = df['Handcap'].astype('int32')
df['Gender'] = df['Gender'].astype('category')

display(df[['Gender', 'ScheduledDay', 'Age', 'No-show']].dtypes)

Gender                     category
ScheduledDay    datetime64[us, UTC]
Age                           int64
No-show                        bool
dtype: object

# Ingeniería de variables

In [5]:
# 1. Calcular el tiempo de espera en días (WaitingDays) utilizando las columnas de fecha originales de tipo datetime
df['WaitingDays'] = (df['AppointmentDay'].dt.normalize() - df['ScheduledDay'].dt.normalize()).dt.days

# 2. Obtener el día de la semana de la cita
df['AppointmentDayOfWeek'] = df['AppointmentDay'].dt.day_name().astype('category')

# 3. Sumar condiciones médicas (comorbilidades)
# Convertimos los booleanos a enteros (0 o 1) para la suma
df['Num_Conditions'] = (
    df['Hipertension'].astype(int) +
    df['Diabetes'].astype(int) +
    df['Alcoholism'].astype(int) +
    (df['Handcap'] > 0).astype(int) # Consideramos si tiene al menos una discapacidad
)

# Mostrar vista previa con las nuevas variables creadas
print("--- Nuevas variables creadas con éxito ---")
display(df[['ScheduledDay', 'AppointmentDay', 'WaitingDays', 'AppointmentDayOfWeek', 'Num_Conditions']].head())
print("\nDimensiones actuales del dataset:", df.shape)

--- Nuevas variables creadas con éxito ---


,ScheduledDay,AppointmentDay,WaitingDays,AppointmentDayOfWeek,Num_Conditions
0,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00,0,Friday,1
1,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00,0,Friday,0
2,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00,0,Friday,0
3,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00,0,Friday,0
4,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00,0,Friday,2



Dimensiones actuales del dataset: (110521, 17)


# Seudonimización de pacientes

Sustituimos `PatientId` por una etiqueta secuencial asignada en el orden de primera aparición tras la limpieza. Para el mismo archivo de entrada, cada visita del mismo paciente queda asociada a la misma etiqueta. Así podemos construir su historial y mantener juntas sus citas durante la validación por grupos. Eliminamos el identificador transaccional `AppointmentID` antes de exportar.

In [6]:
# 1. Crear un mapeo único para pseudo-anonimizar PatientId de forma consistente
unique_patients = df['PatientId'].unique()
patient_id_map = {original_id: f"Paciente_{i+1}" for i, original_id in enumerate(unique_patients)}

# Aplicar el mapeo al DataFrame
df['PatientId'] = df['PatientId'].map(patient_id_map).astype('category')

# 2. Eliminar el identificador directo AppointmentID
if 'AppointmentID' in df.columns:
    df = df.drop(columns=['AppointmentID'])

# Mostrar vista previa de los datos anonimizados
print("--- Dataset Anonimizado con Éxito ---")
display(df[['PatientId', 'Gender', 'Age', 'Neighbourhood', 'No-show']].head())

--- Dataset Anonimizado con Éxito ---


,PatientId,Gender,Age,Neighbourhood,No-show
0,Paciente_1,F,62,JARDIM DA PENHA,False
1,Paciente_2,M,56,JARDIM DA PENHA,False
2,Paciente_3,F,62,MATA DA PRAIA,False
3,Paciente_4,F,8,PONTAL DE CAMBURI,False
4,Paciente_5,F,56,JARDIM DA PENHA,False


# Dataset final preprocesado

In [7]:
df.head()

,PatientId,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show,WaitingDays,AppointmentDayOfWeek,Num_Conditions
0,Paciente_1,F,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00,62,JARDIM DA PENHA,False,True,False,False,0,False,False,0,Friday,1
1,Paciente_2,M,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,False,False,False,False,0,False,False,0,Friday,0
2,Paciente_3,F,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00,62,MATA DA PRAIA,False,False,False,False,0,False,False,0,Friday,0
3,Paciente_4,F,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00,8,PONTAL DE CAMBURI,False,False,False,False,0,False,False,0,Friday,0
4,Paciente_5,F,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,False,True,True,False,0,False,False,0,Friday,2


In [8]:
columnas_esperadas = [
    "PatientId", "Gender", "ScheduledDay", "AppointmentDay", "Age", "Neighbourhood",
    "Scholarship", "Hipertension", "Diabetes", "Alcoholism", "Handcap", "SMS_received",
    "No-show", "WaitingDays", "AppointmentDayOfWeek", "Num_Conditions",
]
assert df.columns.tolist() == columnas_esperadas
assert df.shape == (110521, 16)
assert df["PatientId"].nunique() == 62298
assert df["No-show"].sum() == 22314
assert df[columnas_esperadas].notna().all().all()
assert (df["WaitingDays"] >= 0).all()
print(f"{len(df):,} citas · {df['PatientId'].nunique():,} pacientes · {df['No-show'].sum():,} inasistencias")

110,521 citas · 62,298 pacientes · 22,314 inasistencias


In [9]:
from tempfile import TemporaryDirectory

RUTA_SALIDA = Path("data") / "citas_medicas_preprocesadas.csv"
# Reemplazamos la versión anterior solo al terminar de escribir el CSV.
with TemporaryDirectory(dir=RUTA_SALIDA.parent) as carpeta_temporal:
    provisional = Path(carpeta_temporal) / RUTA_SALIDA.name
    df.to_csv(provisional, index=False, encoding="utf-8", lineterminator="\n")
    provisional.replace(RUTA_SALIDA)
print(f"CSV guardado en {RUTA_SALIDA.as_posix()}.")

CSV guardado en data/citas_medicas_preprocesadas.csv.


El CSV conserva las fechas en formato ISO con zona UTC. Al volver a cargarlo en los notebooks 03, 04 y 05, se deben convertir `ScheduledDay` y `AppointmentDay` con `pd.to_datetime`.